In [0]:
import os
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

load_dotenv()

ADLS_CLIENT_ID       = os.getenv("ADLS_CLIENT_ID")
ADLS_TENANT_ID       = os.getenv("ADLS_TENANT_ID")
ADLS_CLIENT_SECRET   = os.getenv("ADLS_CLIENT_SECRET")
ADLS_STORAGE_ACCOUNT = os.getenv("ADLS_STORAGE_ACCOUNT")
ADLS_CONTAINER       = os.getenv("ADLS_CONTAINER")

print("Credenciais carregadas!")

In [0]:
# Snapshot de referência conforme orientação do escopo
SNAPSHOT_PATH = "vendas_raw/2026/02/21/112200"

# Caminhos completos de cada tabela
TABLES = {
    "ecommerce_categorias"   : f"abfss://{ADLS_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/{SNAPSHOT_PATH}/ecommerce_categorias.parquet",
    "ecommerce_itens_pedido" : f"abfss://{ADLS_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/{SNAPSHOT_PATH}/ecommerce_itens_pedido.parquet",
    "ecommerce_produtos"     : f"abfss://{ADLS_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/{SNAPSHOT_PATH}/ecommerce_produtos.parquet"
}

print(" Caminhos definidos:")
for nome, path in TABLES.items():
    print(f"  Arquivo: {nome}")

In [0]:
import io
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

# Autentica no ADLS
credential = ClientSecretCredential(
    tenant_id     = ADLS_TENANT_ID,
    client_id     = ADLS_CLIENT_ID,
    client_secret = ADLS_CLIENT_SECRET
)

service_client = DataLakeServiceClient(
    account_url = f"https://{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net",
    credential  = credential
)

container_client = service_client.get_file_system_client(ADLS_CONTAINER)

# Lê cada tabela direto na memória com pandas → converte para Spark
import pandas as pd

dataframes = {}

for nome in ["ecommerce_categorias", "ecommerce_itens_pedido", "ecommerce_produtos"]:
    try:
        file_path   = f"{SNAPSHOT_PATH}/{nome}.parquet"
        file_client = container_client.get_file_client(file_path)

        # Baixa direto para memória — sem salvar em disco
        download    = file_client.download_file()
        bytes_data  = download.readall()

        # Lê com pandas via buffer em memória
        pdf = pd.read_parquet(io.BytesIO(bytes_data))

        # Converte para Spark DataFrame
        df = spark.createDataFrame(pdf)
        dataframes[nome] = df

        print(f"OK {nome} → {df.count()} linhas | {len(df.columns)} colunas")

    except Exception as e:
        print(f" Erro ao ler {nome}: {str(e)}")

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when

for nome, df in dataframes.items():
    print("=" * 60)
    print(f"  TABELA: {nome.upper()}")
    print("=" * 60)

    # Schema
    print("\n SCHEMA:")
    df.printSchema()

    # Amostra interativa
    print(f"\n AMOSTRA:")
    display(df)

    # Total de registros
    print(f"\n TOTAL DE REGISTROS: {df.count()}")

    # Nulos por coluna
    print("\n NULOS POR COLUNA:")
    display(
        df.select([
            spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
            for c in df.columns
        ])
    )
    print("\n")